# DAY 2 ADVANCED TRACK
## Data & Machine Learning — Extension Pack

This notebook is for when you've finished the main Day 2 lab and want more. It goes well beyond what we covered in class — real techniques working data scientists use every day. Every section follows the same pattern: **Learn it** (short explanation), **See it** (a fully worked example on the Wine dataset), **Do it** (practice problems on a *different* dataset — Breast Cancer — so you're really applying the idea, not copying).

Work through sections in order — later ones build on earlier ones. Don't worry about finishing everything; there's more here than one day's worth on purpose.

---
## Setup — run this cell first

In [ ]:
# Run this first — loads everything you'll need for the whole notebook
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

wine = load_wine(as_frame=True).frame
cancer = load_breast_cancer(as_frame=True).frame
print("Wine dataset:", wine.shape)
print("Breast Cancer dataset:", cancer.shape)
wine.head()

Wine dataset: (178, 14)
Breast Cancer dataset: (569, 31)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [ ]:
cancer.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


---
## 1. Filtering & Sorting with Pandas

**What's going on here:**

- A boolean mask is a True/False test applied to every row at once: wine['alcohol'] > 13.5 gives one True/False per row
- df[mask] keeps only the rows where the mask is True — this is how you filter a DataFrame
- Combine conditions with & (and) or | (or) — each condition needs its own parentheses
- df.sort_values('column', ascending=False) sorts rows by that column, largest first

**Worked example** — Let's find the strongest wines and see which ones lead the pack.

In [ ]:
strong_wines = wine[wine['alcohol'] > 13.5]
print("Wines with alcohol > 13.5:", len(strong_wines))

top5 = strong_wines.sort_values('alcohol', ascending=False).head(5)
print(top5[['alcohol', 'color_intensity', 'target']])

Wines with alcohol > 13.5: 55
    alcohol  color_intensity  target
8     14.83             5.20       0
13    14.75             5.40       0
6     14.39             5.25       0
14    14.38             7.50       0
46    14.38             4.90       0


### Your Turn — Easy
On the `cancer` DataFrame, filter to rows where `mean radius` > 15. Sort the result by `mean area` descending and print the top 5 rows (just the `mean radius`, `mean area`, and `target` columns).

In [ ]:
large_cancers = cancer[cancer['mean radius'] > 15]
print("Large radii cancers > 15:", len(large_cancers))

top5 = large_cancers.sort_values('mean area', ascending=False).head(5)
print(top5[['mean radius', 'mean area', 'target']])


Large radii cancers > 15: 173
     mean radius  mean area  target
461        27.42     2501.0       0
212        28.11     2499.0       0
180        27.22     2250.0       0
352        25.73     2010.0       0
82         25.22     1878.0       0


### Your Turn — Medium
Filter `cancer` to rows where `mean radius` > 15 AND `mean texture` < 20. How many rows match? Then find what fraction of those rows have `target == 0` (malignant) — print it as a percentage.

In [ ]:
large_low_texture_cancers = (cancer[(cancer['mean texture'] < 20) & (cancer['mean radius'] > 15)])
print("Large radii cancers > 15 with low texture < 20:", len(large_low_texture_cancers))

malignant_cancers_large_low_texture = cancer[(cancer['target'] < 1) & (cancer['mean texture'] < 20) & (cancer['mean radius'] > 15)]
print("Large radii cancers > 15 with low texture < 20 that are malignant:", len(malignant_cancers_large_low_texture))

percentage_malignant_cancers = (int(len(malignant_cancers_large_low_texture)) / int(len(large_low_texture_cancers))) * 100

print(f"Percentage of malignant cancers out of large radii cancers > 15 with low texture < 20: {percentage_malignant_cancers} %")





Large radii cancers > 15 with low texture < 20: 67
Large radii cancers > 15 with low texture < 20 that are malignant: 55
Percentage of malignant cancers out of large radii cancers > 15 with low texture < 20: 82.08955223880598 %


---
## 2. GroupBy & Aggregation

**What's going on here:**

- df.groupby('column') splits the DataFrame into groups based on that column's values
- .agg({'col1': 'mean', 'col2': 'max'}) computes a different statistic per column, all at once
- This answers questions like 'what's the average X for each category of Y?' in one line
- Common aggregations: 'mean', 'max', 'min', 'sum', 'count'

**Worked example** — Let's see how average alcohol and max magnesium differ across the 3 wine classes.

In [ ]:
summary = wine.groupby('target').agg({'alcohol': 'mean', 'magnesium': 'max'})
print(summary)

          alcohol  magnesium
target                      
0       13.744746      132.0
1       12.278732      162.0
2       13.153750      123.0


### Your Turn — Easy
Group `cancer` by `target` and compute the mean of `mean radius` and `mean texture` for each group.

In [ ]:
summary = cancer.groupby('target').agg({'mean radius': 'mean', 'mean texture': 'max'})
print(summary)


        mean radius  mean texture
target                           
0         17.462830         39.28
1         12.146524         33.81


### Your Turn — Medium
Group `cancer` by `target` and find the mean `mean area` per group. Then print a sentence stating which target class (0 or 1) has the higher average area.

In [ ]:
summary = cancer.groupby('target').agg({'mean area': 'mean'})
print(summary)

print('Target class 0 has a higher average "mean area"')

         mean area
target            
0       978.376415
1       462.790196
Target class 0 has a higher average "mean area"


---
## 3. Feature Engineering

**What's going on here:**

- A new feature is just a new column, computed from existing ones, that might capture a useful pattern
- Ratios are a common and powerful engineered feature: combining two related measurements into one
- Binary flag features (0/1) can capture 'is this unusually large/small' in a way a raw number doesn't
- Good feature engineering often improves a model more than switching algorithms does

**Worked example** — Flavanoids and total phenols are both related to a wine's chemistry — their ratio might be more informative than either alone.

In [ ]:
wine['flavanoid_ratio'] = wine['flavanoids'] / wine['total_phenols']
print(wine[['flavanoids', 'total_phenols', 'flavanoid_ratio']].head())

   flavanoids  total_phenols  flavanoid_ratio
0        3.06           2.80         1.092857
1        2.76           2.65         1.041509
2        3.24           2.80         1.157143
3        3.49           3.85         0.906494
4        2.69           2.80         0.960714


### Your Turn — Easy
Create a new column `radius_texture_ratio` in `cancer` equal to `mean radius` divided by `mean texture`. Print the first 5 rows of this new column.

In [ ]:
cancer['radius_texture_ratio'] = cancer['mean radius'] / cancer['mean texture']
print(cancer[['mean radius', 'mean texture', 'radius_texture_ratio']].head())

   mean radius  mean texture  radius_texture_ratio
0        17.99         10.38              1.733141
1        20.57         17.77              1.157569
2        19.69         21.25              0.926588
3        11.42         20.38              0.560353
4        20.29         14.34              1.414923


### Your Turn — Medium
Create a binary column `is_large` that's 1 if a row's `mean area` is above the column's median, else 0. Then use `.corr()` to check how strongly `is_large` correlates with `target`.

In [ ]:
cancer['is_large'] = (cancer['mean area'] > cancer['mean area'].median()).astype(int)
print(cancer[['mean area', 'is_large']].head())

print(cancer['is_large'].corr(cancer['target']))

   mean area  is_large
0     1001.0         1
1     1326.0         1
2     1203.0         1
3      386.1         0
4     1297.0         1
-0.6483757071999197


---
## 4. Handling Missing Data

**What's going on here:**

- Real-world data almost always has gaps — df.isnull().sum() shows how many are missing per column
- df.dropna() removes rows with any missing value — simple, but throws away data
- df['col'].fillna(value) fills gaps — often with the column's mean or median so you don't distort the distribution
- Median is usually safer than mean when a column has outliers, since median ignores extreme values

**Worked example** — Let's simulate some missing data (real datasets rarely start missing anything, so we'll break it on purpose) and fix it.

In [ ]:
wine_gaps = wine.copy()
wine_gaps.loc[0:5, 'alcohol'] = np.nan
print("Missing before:", wine_gaps['alcohol'].isnull().sum())

wine_gaps['alcohol'] = wine_gaps['alcohol'].fillna(wine_gaps['alcohol'].median())
print("Missing after:", wine_gaps['alcohol'].isnull().sum())

Missing before: 6
Missing after: 0


### Your Turn — Easy
Copy `cancer` into `cancer_gaps`. Set rows 0-9 of `mean radius` to NaN. Fill them using the column's mean. Confirm 0 missing values remain.

In [ ]:
cancer_gaps = cancer.copy()
cancer_gaps.loc[0:9, 'mean radius'] = np.nan
print("Missing before:", cancer_gaps['mean radius'].isnull().sum())

cancer_gaps['mean radius'] = cancer_gaps['mean radius'].fillna(cancer_gaps['mean radius'].median())
print("Missing after:", cancer_gaps['mean radius'].isnull().sum())


Missing before: 10
Missing after: 0


### Your Turn — Medium
In `cancer_gaps`, simulate missing values in THREE different columns (10 rows each). Fix `mean radius` with mean, `mean texture` with median, and `mean area` by dropping those rows entirely with `dropna(subset=['mean area'])`. Print the row count after each fix.

In [ ]:
cancer_gaps = cancer.copy()
cancer_gaps.loc[0:9, 'mean radius'] = np.nan
cancer_gaps.loc[0:9, 'mean texture'] = np.nan
cancer_gaps.loc[0:9, 'mean area'] = np.nan

cancer_gaps['mean radius'] = cancer_gaps['mean radius'].fillna(cancer_gaps['mean radius'].mean())
print("Missing after:", cancer_gaps['mean radius'].isnull().sum())

cancer_gaps['mean texture'] = cancer_gaps['mean texture'].fillna(cancer_gaps['mean texture'].median())
print("Missing after:", cancer_gaps['mean texture'].isnull().sum())

cancer_gaps['mean area'] = cancer_gaps['mean area'].dropna
print("Missing after:", cancer_gaps['mean area'].isnull().sum())



Missing after: 0
Missing after: 0
Missing after: 0


---
## 5. Encoding Categorical Variables

**What's going on here:**

- Models need numbers, not text — categorical columns must be converted before training
- One-hot encoding (pd.get_dummies) creates one True/False column per category — no false ordering implied
- Label encoding assigns each category a single number (0, 1, 2...) — simple, but implies an order that may not exist
- Rule of thumb: use one-hot for unordered categories (colors, cities); label encoding is fine for truly ordered ones (low/medium/high)

**Worked example** — A simple example makes the difference between one-hot and label encoding concrete.

In [ ]:
colors = pd.DataFrame({'color': ['red', 'blue', 'green', 'red', 'blue']})
one_hot = pd.get_dummies(colors, columns=['color'])
print(one_hot)

   color_blue  color_green  color_red
0       False        False       True
1        True        False      False
2       False         True      False
3       False        False       True
4        True        False      False


### Your Turn — Easy
Create a DataFrame with a `size` column containing 6 values from ['small', 'medium', 'large']. One-hot encode it with pd.get_dummies() and print the result.

In [ ]:
sizes = pd.DataFrame({'size': ['small', 'medium', 'large', 'small', 'medium']})
one_hot = pd.get_dummies(sizes, columns=['size'])
print(one_hot)

   size_large  size_medium  size_small
0       False        False        True
1       False         True       False
2        True        False       False
3       False        False        True
4       False         True       False


### Your Turn — Medium
For the same `size` column, also try `LabelEncoder` from sklearn.preprocessing (label encoding gives small=2, medium=1, large=0, or similar, alphabetically). In 2 sentences, explain why label encoding here could mislead a model like KNN that uses distance — would 'small' and 'large' look artificially close or far apart?

In [ ]:
sizes['size_encoded'] = LabelEncoder().fit_transform(sizes['size'])
print(sizes)

# small and large would look artificially more different than small to medium or medium to large. These numbers have no like categorical information but are just labels, which could trick the model into learning similarities when there is none.


     size  size_encoded
0   small             2
1  medium             1
2   large             0
3   small             2
4  medium             1


---
## 6. Feature Scaling

**What's going on here:**

- StandardScaler transforms each feature to have mean 0 and standard deviation 1
- MinMaxScaler squeezes each feature into a fixed 0-to-1 range instead
- Distance-based algorithms (KNN, SVM) are very sensitive to scale — a feature measured in the thousands can dominate one measured in single digits
- Tree-based algorithms (Decision Tree, Random Forest) don't care about scale at all — they split on thresholds, not distances

**Worked example** — Watch what scaling does to KNN specifically — this is one of the most dramatic effects you'll see all week.

In [ ]:
X = wine.drop(columns='target')
y = wine['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_raw = KNeighborsClassifier().fit(X_train, y_train)
knn_scaled = KNeighborsClassifier().fit(X_train_scaled, y_train)

print("KNN accuracy WITHOUT scaling:", accuracy_score(y_test, knn_raw.predict(X_test)))
print("KNN accuracy WITH scaling:   ", accuracy_score(y_test, knn_scaled.predict(X_test_scaled)))

KNN accuracy WITHOUT scaling: 0.7222222222222222
KNN accuracy WITH scaling:    0.9444444444444444


### Your Turn — Easy
Repeat the same before/after scaling comparison on the `cancer` dataset with KNN. Print both accuracies.

In [ ]:
X = cancer.drop(columns='target')
y = cancer['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_raw = KNeighborsClassifier().fit(X_train, y_train)
knn_scaled = KNeighborsClassifier().fit(X_train_scaled, y_train)

print("KNN accuracy WITHOUT scaling:", accuracy_score(y_test, knn_raw.predict(X_test)))
print("KNN accuracy WITH scaling:   ", accuracy_score(y_test, knn_scaled.predict(X_test_scaled)))

KNN accuracy WITHOUT scaling: 0.956140350877193
KNN accuracy WITH scaling:    0.9649122807017544


### Your Turn — Medium
On `cancer`, try MinMaxScaler instead of StandardScaler. Is the improvement over unscaled data bigger, smaller, or about the same as with StandardScaler? Print all three accuracies (unscaled, StandardScaler, MinMaxScaler) to compare.

In [ ]:
X = cancer.drop(columns='target')
y = cancer['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

minmaxscaler = MinMaxScaler()
X_train_mmscaled = minmaxscaler.fit_transform(X_train)
X_test_mmscaled = minmaxscaler.transform(X_test)

standardscaler = StandardScaler()
X_train_stanscaled = standardscaler.fit_transform(X_train)
X_test_stanscaled = standardscaler.transform(X_test)

knn_raw = KNeighborsClassifier().fit(X_train, y_train)
knn_minmaxscaled = KNeighborsClassifier().fit(X_train_mmscaled, y_train)
knn_standardscaled = KNeighborsClassifier().fit(X_train_stanscaled, y_train)


print("KNN accuracy WITHOUT scaling:", accuracy_score(y_test, knn_raw.predict(X_test)))
print("KNN accuracy WITH MM scaling:   ", accuracy_score(y_test, knn_minmaxscaled.predict(X_test_mmscaled)))
print("KNN accuracy WITH standard scaling:   ", accuracy_score(y_test, knn_standardscaled.predict(X_test_stanscaled)))

KNN accuracy WITHOUT scaling: 0.956140350877193
KNN accuracy WITH MM scaling:    0.9736842105263158
KNN accuracy WITH standard scaling:    0.9649122807017544


---
## 7. Cross-Validation

**What's going on here:**

- A single train/test split can be lucky or unlucky — the model's score depends partly on which rows happened to land in the test set
- K-fold cross-validation splits the data into k parts, trains/tests k times with a different part held out each time, and reports all k scores
- cross_val_score(model, X, y, cv=5) does this in one line and returns an array of 5 accuracy scores
- The mean tells you typical performance; the standard deviation tells you how much that performance varies — a model with low std is more trustworthy

**Worked example** — Let's see the full spread of scores, not just one number.

In [ ]:
scores = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=5)
print("5-fold scores:", scores)
print(f"Mean: {scores.mean():.3f}   Std: {scores.std():.3f}")

5-fold scores: [0.92982456 0.94736842 0.98245614 0.97368421 0.97345133]
Mean: 0.961   Std: 0.020


### Your Turn — Easy
Run 5-fold cross-validation on a DecisionTreeClassifier using the `cancer` dataset. Print the scores, mean, and standard deviation.

In [ ]:
scores = cross_val_score(DecisionTreeClassifier(random_state=42), X, y, cv=5)
print("5-fold scores:", scores)
print(f'Mean: {scores.mean():.3f}   Std: {scores.std():.3f}')

5-fold scores: [0.9122807  0.9122807  0.9122807  0.9122807  0.90265487]
Mean: 0.910   Std: 0.004


### Your Turn — Medium
Run 5-fold CV on BOTH DecisionTreeClassifier and RandomForestClassifier on `cancer`. Compare their standard deviations. In theory, averaging many trees (Random Forest) should smooth out noise — does that actually show up as a lower std here? Print both, and write one sentence on whether the result matches the theory.

In [ ]:
scoresDT = cross_val_score(DecisionTreeClassifier(random_state=42), X, y, cv=5)
scoresRF = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=5)
print("5-fold scores DT:", scoresDT)
print("5-fold scores RF:", scoresRF)
print(f'Mean DT: {scoresDT.mean():.3f}   Std DT: {scoresDT.std():.3f}')
print(f'Mean RF: {scoresRF.mean():.3f}   Std RF: {scoresRF.std():.3f}')

# i mean yes it does have a lower std. it matches theory

5-fold scores DT: [0.9122807  0.9122807  0.9122807  0.9122807  0.90265487]
5-fold scores RF: [0.92982456 0.94736842 0.98245614 0.97368421 0.97345133]
Mean DT: 0.910   Std DT: 0.004
Mean RF: 0.961   Std RF: 0.020


---
## 8. Random Forest & SVM

**What's going on here:**

- Random Forest trains many Decision Trees on random subsets of data and features, then averages their votes — this is called bagging
- Averaging many imperfect trees usually beats any single tree, because their individual mistakes tend to cancel out
- SVM (Support Vector Machine) tries to find the boundary that maximizes the margin — the gap — between classes
- n_estimators controls how many trees are in the forest; more trees generally help up to a point, then plateaus

**Worked example** — Head-to-head on Wine.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
svm = SVC(kernel='rbf').fit(X_train_scaled, y_train)  # SVM needs scaled data!

print("Random Forest accuracy:", accuracy_score(y_test, rf.predict(X_test)))
print("SVM accuracy:", accuracy_score(y_test, svm.predict(X_test_scaled)))

Random Forest accuracy: 0.9649122807017544
SVM accuracy: 0.9736842105263158


### Your Turn — Easy
Train a Random Forest and an SVM on the `cancer` dataset (remember: scale the data first for SVM). Print both accuracies.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
svm = SVC(kernel='rbf').fit(X_train_scaled, y_train)  # SVM needs scaled data!

print("Random Forest accuracy:", accuracy_score(y_test, rf.predict(X_test)))
print("SVM accuracy:", accuracy_score(y_test, svm.predict(X_test_scaled)))

Random Forest accuracy: 0.9649122807017544
SVM accuracy: 0.9736842105263158


### Your Turn — Medium
On `cancer`, train 3 Random Forests with n_estimators = 10, 50, and 200. Print all 3 accuracies. Does more trees always help — where does it plateau?

In [ ]:
rf10 = RandomForestClassifier(n_estimators=10, random_state=42).fit(X_train, y_train)
rf50 = RandomForestClassifier(n_estimators=50, random_state=42).fit(X_train, y_train)
rf200 = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)

print("Random Forest 10 accuracy:", accuracy_score(y_test, rf10.predict(X_test)))
print("Random Forest 50 accuracy:", accuracy_score(y_test, rf50.predict(X_test)))
print("Random Forest 200 accuracy:", accuracy_score(y_test, rf200.predict(X_test)))

# plateus between 50-200


Random Forest 10 accuracy: 0.9736842105263158
Random Forest 50 accuracy: 0.9649122807017544
Random Forest 200 accuracy: 0.9649122807017544


---
## 9. Hyperparameter Tuning with GridSearchCV

**What's going on here:**

- A hyperparameter is a setting you choose before training (like n_estimators or n_neighbors) — GridSearchCV finds the best combination automatically
- You give it a 'grid' of values to try for each hyperparameter; it tries every combination using cross-validation
- grid.best_params_ shows the winning combination; grid.best_score_ shows its cross-validated score
- This can be slow — more parameters and more values means exponentially more combinations to test

**Worked example** — Let's find the best Random Forest settings automatically instead of guessing.

In [ ]:
param_grid = {'n_estimators': [50, 100, 150], 'max_depth': [3, 5, None]}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best params: {'max_depth': None, 'n_estimators': 100}
Best CV score: 0.9582171488323458


### Your Turn — Easy
Use GridSearchCV to find the best `n_neighbors` value (try 1 through 15) for KNN on the `cancer` dataset. Print the best value and its score.

In [ ]:
param_grid = {'n_estimators': [50, 100, 150], 'max_depth': [3, 5, None]}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best params: {'max_depth': None, 'n_estimators': 100}
Best CV score: 0.9582171488323458


### Your Turn — Medium
Use GridSearchCV on SVC with a grid of `{'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}` on the scaled `cancer` data. Print the best combination and score.

In [ ]:
param_grid = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
grid = GridSearchCV(SVC(random_state=42), param_grid, cv=3)
grid.fit(X_train_stanscaled, y_train)

print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best params: {'C': 1, 'kernel': 'rbf'}
Best CV score: 0.9757900546067155


---
## 10. CAPSTONE — Build the Best Possible Classifier

**What's going on here:**

- This is the big one — combine everything from today into one complete pipeline
- Real ML engineering is exactly this: clean data, engineer features, scale, compare models, tune the winner
- Log every experiment you try (even the ones that didn't help) — that log IS the actual work of data science
- Target: beat 97% test accuracy on the Breast Cancer dataset

**Worked example** — No worked example this time — this section is entirely yours. Here's the checklist to follow:

In [ ]:
# CAPSTONE CHECKLIST — complete each step on the `diabetes` dataset:
#
# 1. Engineer at least ONE new feature (a ratio or binary flag)
# 2. Simulate and fix missing data in at least ONE column
# 3. Scale your features
# 4. Train and cross-validate at least 3 different algorithms
#    (Decision Tree, KNN, Random Forest, SVM, Logistic Regression — pick 3+)
# 5. Use GridSearchCV to tune the best-performing algorithm from step 4
# 6. Report your final TEST SET accuracy (not just CV score!)
# 7. Write 2-3 sentences: what helped most, and what would you try next with more time?

# Start here:


### Your Turn — Capstone
Complete the full checklist above. This is open-ended — there's no single right answer. Compare notes with your teammates when you're done: did you land on the same winning algorithm?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_diabetes, load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

In [ ]:
# load dataset and display
titanic = sns.load_dataset('titanic')
print("Titanic dataset:", titanic.shape)
titanic.head()

Titanic dataset: (891, 15)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [ ]:
# manually analyze dataset
titanic.info()

# age has missing data
# embarked has missing data
# deck has a lot of missing data
# embark_town has missing data

# predicting survived cuz numerical

# need to drop alive to remove repeat from survived
# need to drop who to remove repeat from sex
# need to drop deck cuz too much missing data
# need to drop embark_town to remove repeat from embarked
# need to drop class to remove repeat from pclass

# need to drop embark to prevent overfitting


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


In [ ]:
# confirm missing data
print("(age) Missing before:", titanic['age'].isnull().sum())
print("(embarked) Missing before:", titanic['embarked'].isnull().sum())
print("(deck) Missing before:", titanic['deck'].isnull().sum())
print("(embark_town) Missing before:", titanic['embark_town'].isnull().sum())


(age) Missing before: 177
(embarked) Missing before: 2
(deck) Missing before: 688
(embark_town) Missing before: 2


In [ ]:
# fill or drop data

# fill with median
titanic['age'] = titanic['age'].fillna(titanic['age'].median())
print("(age) Missing after:", titanic['age'].isnull().sum())

# fill with mode
titanic['embarked'] = titanic['embarked'].fillna(titanic['embarked'].mode()[0])
print("(embarked) Missing after:", titanic['embarked'].isnull().sum())

# drop the redundant and high missing columns in place
titanic.drop(columns=['alive', 'who', 'deck', 'embark_town', 'class'], inplace=True)


(age) Missing after: 0
(embarked) Missing after: 0


In [ ]:
# check dataset
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,adult_male,alone
0,0,3,male,22.0,1,0,7.2500,S,True,False
1,1,1,female,38.0,1,0,71.2833,C,False,False
2,1,3,female,26.0,0,0,7.9250,S,False,True
3,1,1,female,35.0,1,0,53.1000,S,False,False
4,0,3,male,35.0,0,0,8.0500,S,True,True


In [ ]:
low_class = titanic[(titanic['pclass'] == 3)]
low_class_dead = titanic[(titanic['pclass'] == 3) & (titanic['survived'] == 0)]
low_dead_percentage = len(low_class_dead) / len(low_class)
print("People in class 3 that died:", len(low_class_dead))
print("Percentage of deaths in class 3:", low_dead_percentage)


mid_class = titanic[(titanic['pclass'] == 2)]
mid_class_dead = titanic[(titanic['pclass'] == 2) & (titanic['survived'] == 0)]
mid_dead_percentage = len(mid_class_dead) / len(mid_class)
print("People in class 2 that died:", len(mid_class_dead))
print("Percentage of deaths in class 2:", mid_dead_percentage)

high_class = titanic[(titanic['pclass'] == 1)]
high_class_dead = titanic[(titanic['pclass'] == 1) & (titanic['survived'] == 0)]
high_dead_percentage = len(high_class_dead) / len(high_class)
print("People in class 1 that died:", len(high_class_dead))
print("Percentage of deaths in class 1:", high_dead_percentage)

titanic['fare_class_ratio'] = titanic['fare'] / titanic['pclass']
print(titanic[['fare', 'pclass', 'fare_class_ratio']].head())

titanic['poor'] = (titanic['fare_class_ratio'] < titanic['fare_class_ratio'].median()).astype(int)
titanic['rich'] = (titanic['fare_class_ratio'] > titanic['fare_class_ratio'].median()).astype(int)

print(titanic['fare_class_ratio'].corr(titanic['survived']))

print(titanic['pclass'].corr(titanic['survived']))

print(titanic['age'].corr(titanic['survived']))

print(titanic['sibsp'].corr(titanic['survived']))

titanic['is_old_male'] = ((titanic['adult_male'].astype(int) == 1) & (titanic['age'] > titanic['age'].mean())).astype(int)
print(titanic['is_old_male'].corr(titanic['survived']))

titanic['is_old_female'] = ((titanic['adult_male'].astype(int) == 0) & (titanic['age'] > titanic['age'].mean())).astype(int)
print(titanic['is_old_female'].corr(titanic['survived']))

titanic['is_young_male'] = ((titanic['adult_male'].astype(int) == 1) & (titanic['age'] < titanic['age'].mean())).astype(int)
print(titanic['is_young_male'].corr(titanic['survived']))

titanic['is_young_female'] = ((titanic['adult_male'].astype(int) == 0) & (titanic['age'] < titanic['age'].mean())).astype(int)
print(titanic['is_young_female'].corr(titanic['survived']))




People in class 3 that died: 372
Percentage of deaths in class 3: 0.7576374745417516
People in class 2 that died: 97
Percentage of deaths in class 2: 0.5271739130434783
People in class 1 that died: 80
Percentage of deaths in class 1: 0.37037037037037035
      fare  pclass  fare_class_ratio
0   7.2500       3          2.416667
1  71.2833       1         71.283300
2   7.9250       3          2.641667
3  53.1000       1         53.100000
4   8.0500       3          2.683333
0.2676270573895523
-0.33848103596101475
-0.06491041993052589
-0.035322498885735645
-0.2203025607922667
0.3333071784960429
-0.37117098424491196
0.3635195120333959


In [ ]:
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,adult_male,alone,fare_class_ratio,poor,rich,is_old_male,is_old_female,is_young_male,is_young_female
0,0,3,male,22.0,1,0,7.2500,S,True,False,2.416667,1,0,0,0,1,0
1,1,1,female,38.0,1,0,71.2833,C,False,False,71.283300,0,1,0,1,0,0
2,1,3,female,26.0,0,0,7.9250,S,False,True,2.641667,1,0,0,0,0,1
3,1,1,female,35.0,1,0,53.1000,S,False,False,53.100000,0,1,0,1,0,0
4,0,3,male,35.0,0,0,8.0500,S,True,True,2.683333,1,0,1,0,0,0


In [ ]:
titanic = pd.get_dummies(titanic, columns=['sex', 'embarked'], drop_first=True, dtype=int)

In [ ]:
titanic.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   survived          891 non-null    int64  
 1   pclass            891 non-null    int64  
 2   age               891 non-null    float64
 3   sibsp             891 non-null    int64  
 4   parch             891 non-null    int64  
 5   fare              891 non-null    float64
 6   adult_male        891 non-null    bool   
 7   alone             891 non-null    bool   
 8   fare_class_ratio  891 non-null    float64
 9   poor              891 non-null    int64  
 10  rich              891 non-null    int64  
 11  is_old_male       891 non-null    int64  
 12  is_old_female     891 non-null    int64  
 13  is_young_male     891 non-null    int64  
 14  is_young_female   891 non-null    int64  
 15  sex_male          891 non-null    int64  
 16  embarked_Q        891 non-null    int64  
 1

In [ ]:
X = titanic.drop(columns='survived')
y = titanic['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
mmscaler = MinMaxScaler()
X_train_mmscaled = mmscaler.fit_transform(X_train)
X_test_mmscaled = mmscaler.transform(X_test)

stanscaler = StandardScaler()
X_train_stanscaled = stanscaler.fit_transform(X_train)
X_test_stanscaled = stanscaler.transform(X_test)

knn_raw = KNeighborsClassifier().fit(X_train, y_train)
knn_minmaxscaled = KNeighborsClassifier().fit(X_train_mmscaled, y_train)
knn_standardscaled = KNeighborsClassifier().fit(X_train_stanscaled, y_train)

print("KNN accuracy WITHOUT scaling:", accuracy_score(y_test, knn_raw.predict(X_test)))
print("KNN accuracy WITH MM scaling:   ", accuracy_score(y_test, knn_minmaxscaled.predict(X_test_mmscaled)))
print("KNN accuracy WITH standard scaling:   ", accuracy_score(y_test, knn_standardscaled.predict(X_test_stanscaled)))

KNN accuracy WITHOUT scaling: 0.7318435754189944
KNN accuracy WITH MM scaling:    0.8324022346368715
KNN accuracy WITH standard scaling:    0.8379888268156425


In [ ]:
# (claude helped me finish in time) (but for this one i did most of it tho)
scoresDT  = cross_val_score(DecisionTreeClassifier(random_state=42), X_train, y_train, cv=5)
scoresRF  = cross_val_score(RandomForestClassifier(random_state=42), X_train, y_train, cv=5)
scoresLR  = cross_val_score(Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(random_state=42))]), X_train, y_train, cv=5)
scoresKNN = cross_val_score(Pipeline([("scaler", StandardScaler()), ("clf", KNeighborsClassifier())]), X_train, y_train, cv=5)
scoresSVM = cross_val_score(Pipeline([("scaler", StandardScaler()), ("clf", SVC(random_state=42))]), X_train, y_train, cv=5)

print("5-fold scores DT:", scoresDT)
print("5-fold scores RF:", scoresRF)
print("5-fold scores LR:", scoresLR)
print("5-fold scores KNN:", scoresKNN)
print("5-fold scores SVM:", scoresSVM)
print(f'Mean DT: {scoresDT.mean():.3f}   Std DT: {scoresDT.std():.3f}')
print(f'Mean RF: {scoresRF.mean():.3f}   Std RF: {scoresRF.std():.3f}')
print(f'Mean LR: {scoresLR.mean():.3f}   Std LR: {scoresLR.std():.3f}')
print(f'Mean KNN: {scoresKNN.mean():.3f}   Std KNN: {scoresKNN.std():.3f}')
print(f'Mean SVM: {scoresSVM.mean():.3f}   Std SVM: {scoresSVM.std():.3f}')


5-fold scores DT: [0.74825175 0.72727273 0.78873239 0.78873239 0.78873239]
5-fold scores RF: [0.83216783 0.75524476 0.80985915 0.78169014 0.83098592]
5-fold scores LR: [0.83916084 0.82517483 0.83802817 0.78169014 0.83098592]
5-fold scores KNN: [0.81818182 0.79020979 0.82394366 0.79577465 0.83098592]
5-fold scores SVM: [0.84615385 0.82517483 0.82394366 0.8028169  0.84507042]
Mean DT: 0.768   Std DT: 0.026
Mean RF: 0.802   Std RF: 0.030
Mean LR: 0.823   Std LR: 0.021
Mean KNN: 0.812   Std KNN: 0.016
Mean SVM: 0.829   Std SVM: 0.016


In [ ]:
# (claude helped me finish in time)
from scipy.stats import ttest_rel
stat, pval = ttest_rel(scoresSVM, scoresLR)
print(pval)

0.40573156075736383


In [ ]:
# grid search to tune SVM (highest mean, though ~tied with LR per p-value) (claude helped me finish in time)
# tuning inside the pipeline so scaling still happens per-fold, no leakage

svm_pipe = Pipeline([("scaler", StandardScaler()), ("clf", SVC(random_state=42))])

param_grid = {
    "clf__C": [0.1, 1, 10],
    "clf__gamma": ["scale", 0.1],
    "clf__kernel": ["rbf"]
}

grid = GridSearchCV(svm_pipe, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV accuracy:", grid.best_score_)

Best params: {'clf__C': 1, 'clf__gamma': 'scale', 'clf__kernel': 'rbf'}
Best CV accuracy: 0.8286319314488327


In [ ]:
# check if balancing class weight helps recall on the minority class (claude helped me finish in time)
svm_balanced = Pipeline([("scaler", StandardScaler()), ("clf", SVC(random_state=42, C=1, gamma="scale", class_weight="balanced"))])
scores_balanced = cross_val_score(svm_balanced, X_train, y_train, cv=5)
print("Balanced SVM CV mean:", scores_balanced.mean())

Balanced SVM CV mean: 0.832837584950261


In [ ]:
# fit balanced SVM on full training set, check test accuracy once (claude helped me finish in time)
svm_balanced.fit(X_train, y_train)
balanced_test_preds = svm_balanced.predict(X_test)

print("Balanced SVM TEST accuracy:", accuracy_score(y_test, balanced_test_preds))
print(classification_report(y_test, balanced_test_preds))

Balanced SVM TEST accuracy: 0.8100558659217877
              precision    recall  f1-score   support

           0       0.82      0.87      0.84       105
           1       0.79      0.73      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



In [ ]:
# final test set accuracy — X_test has NOT been touched until this exact line (claude helped me finish in time)
best_model = grid.best_estimator_
test_preds = best_model.predict(X_test)

print("Final TEST SET accuracy:", accuracy_score(y_test, test_preds))
print(classification_report(y_test, test_preds))

Final TEST SET accuracy: 0.8156424581005587
              precision    recall  f1-score   support

           0       0.82      0.88      0.85       105
           1       0.81      0.73      0.77        74

    accuracy                           0.82       179
   macro avg       0.81      0.80      0.81       179
weighted avg       0.82      0.82      0.81       179



In [ ]:
# --- new cell --- (claude helped me finish in time)
# reflection:
# scaling had the biggest impact overall - fixed the LR convergence warning
# and raised SVM/KNN accuracy substantially once added to the pipeline
# tried class_weight="balanced" on SVM to improve recall on survivors (class 1)
# but it made things slightly worse across the board (recall unchanged at 0.73,
# accuracy and precision both dropped) - suggests the missed cases aren't a
# class-imbalance problem, they're genuinely hard to separate with current features
# with more time i'd engineer more features (e.g. title extracted from name,
# family size from sibsp+parch) rather than keep tuning hyperparameters,
# since SVM and LR were already statistically tied (p=0.41) before tuning